## Importación de módulos

In [7]:
import torch
import torch.nn as nn
import math
from transformers import BertForSequenceClassification, BertTokenizer

### Implementación de Clase basada en polinomios de Chebishev

In [5]:
class ChebyshevPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super(ChebyshevPositionalEncoding, self).__init__()
        self.d_model = d_model
        self.max_len = max_len

        pe = torch.zeros(max_len, d_model)

        if max_len > 1:
            p = torch.arange(0, max_len, dtype=torch.float32)
            x = -1.0 + 2.0 * p / (max_len - 1)
        else:
            x = torch.zeros(1, dtype=torch.float32)
            
        arccos_x = torch.acos(x)  
        k = torch.arange(0, d_model, dtype=torch.float32) 
        
        angles = arccos_x.unsqueeze(1) * k.unsqueeze(0)
        pe = torch.cos(angles)
        
        pe = pe / math.sqrt(d_model)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x_input):
        seq_len = x_input.size(1)
        return x_input + self.pe[:, :seq_len, :]

### Configuración del Dispositivo y Carga del Modelo Preentrenado

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Ejecutando en: cpu


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
